# 01 · Conocer y preparar los datos

**Proyecto Unidad 2 — Inferencia estadística aplicada a un problema de Ciencia de Datos**
**Tema:** Deserción (*churn*) de clientes de un banco europeo

---

### ¿Qué se hace en este cuaderno?

Antes de calcular cualquier cosa hay que **conocer bien los datos**: de dónde vienen, qué significa cada columna y si son confiables. Este cuaderno tiene cuatro pasos:

| Paso | Qué se hace | Por qué |
|---|---|---|
| 1 | Ficha técnica del dataset | Saber de dónde vienen los datos y qué limitaciones tienen |
| 2 | Primer vistazo y diccionario de variables | Entender qué mide cada columna |
| 3 | Revisión de calidad | Confirmar que no hay errores que arruinen las conclusiones |
| 4 | Preparación | Dejar los datos listos (y en español) para los siguientes cuadernos |

> 💡 **Idea clave:** más adelante daremos resultados como rangos ("entre 18 % y 23 %"). Esos rangos sólo toman en cuenta el error que viene de **revisar a unos clientes y no a todos**. No pueden corregir datos mal capturados: si hubiera clientes repetidos o edades imposibles, el rango quedaría "bonito" pero alrededor de un valor equivocado. Por eso, **primero se revisan los datos**.
>
> 📘 ¿Es tu primera vez con estadística? Empieza por `00_aprende_desde_cero.ipynb`: explica todas las ideas desde cero.

> 📄 **En el reporte PDF:** este cuaderno cubre la «Ficha técnica y justificación del dataset» (Tabla 1), la sección **2.1 Población y calidad de los datos** y la preparación descrita al inicio de la sección **3**. Los pasos de este cuaderno no son las secciones del reporte: por eso se llaman «Paso 1..4».



## Paso 1 · Ficha técnica del dataset

> 📄 **En el reporte PDF:** «Ficha técnica y justificación del dataset», Tabla 1, párrafos «¿Por qué este dataset?», «Ética y privacidad» y «Limitaciones de la fuente».


| Campo | Descripción |
|---|---|
| **Nombre** | *Churn Modelling* (archivo `Churn_Modelling.csv`) |
| **Fuente** | Kaggle, repositorio público de datasets para ciencia de datos |
| **Qué contiene** | 10,000 clientes de un banco, uno por fila, con su perfil y si abandonaron el banco |
| **Alcance geográfico** | **Europa:** clientes de **Francia, Alemania y España** |
| **Lo que queremos estudiar** | La columna `Exited`: 1 = el cliente cerró su cuenta y se fue (abandonó), 0 = se quedó |
| **Moneda** | No viene indicada. Como los tres países usan el **euro**, los montos se interpretan en euros (€) |
| **Periodo** | No viene documentado. Se interpreta como una fotografía de la cartera con el estatus de abandono de cada cliente |
| **Privacidad** | Datos **anonimizados** (no se puede saber quién es cada persona): el cliente se identifica con un número interno y sólo aparece su apellido, sin nombre, dirección ni documentos |

### ¿Por qué se eligió este dataset?

1. **Son datos de clientes de un banco, no simulados por el equipo.** Esto hace que las conclusiones describan un comportamiento observado en una cartera real de clientes.
2. **Es un dataset de referencia:** es uno de los conjuntos de datos de *churn* bancario más usados en cursos y trabajos de ciencia de datos, por lo que otros pueden revisar y reproducir el análisis.
3. **Tiene justo las variables que el problema necesita:** abandono (sí/no), saldo, edad, actividad del cliente, país, productos contratados.
4. **El tamaño es adecuado:** 10,000 clientes permiten seleccionar una muestra y, además, comparar los resultados con el total.

### Consideraciones éticas y de privacidad (Unión Europea)

Los clientes viven en países de la Unión Europea, donde existe una ley muy estricta para cuidar los datos personales: el **Reglamento General de Protección de Datos (RGPD)**. Una de sus reglas es la **minimización de datos**: *guarda sólo los datos que de verdad necesitas*. Por eso se borra la columna `Surname` (apellido): **ninguna pregunta del estudio la necesita**. Además, los resultados se dan sólo en conjunto (porcentajes y promedios), nunca de un cliente en particular.

### Limitaciones de la fuente (se reportan desde el inicio)

- El banco de origen **no está identificado** y la fuente no documenta cómo ni cuándo se recolectaron los datos.
- No hay fechas: no se sabe en qué periodo ocurrió el abandono.
- Por eso, las conclusiones aplican a **este grupo de clientes** (la "cartera" del banco), no a "todos los bancos europeos".

In [1]:
# QUÉ HACE ESTA CELDA: Carga pandas, la herramienta para trabajar con tablas
# pandas: trabajar con tablas, como en Excel
import pandas as pd

## Paso 2 · Primer vistazo

> 📄 **En el reporte PDF:** no tiene sección propia; sirve para conocer las columnas antes de revisarlas.



In [2]:
# QUÉ HACE ESTA CELDA: Lee el archivo original y cuenta filas y columnas
# read_csv lee el archivo CSV y lo guarda como tabla
datos = pd.read_csv("../data/raw/Churn_Modelling.csv")
print("Filas (clientes):", len(datos))
# shape[1] es el número de columnas
print("Columnas:", datos.shape[1])

Filas (clientes): 10000
Columnas: 14


In [3]:
# QUÉ HACE ESTA CELDA: Muestra las primeras 5 filas para ver cómo son los datos
# head() muestra sólo las primeras filas
datos.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


### Diccionario de variables

| Columna original | Nombre en español | Tipo | Qué significa |
|---|---|---|---|
| `RowNumber` | — | — | Número de renglón del archivo (no aporta información) |
| `CustomerId` | `id_cliente` | Identificador | Número interno del cliente |
| `Surname` | — | Dato personal | Apellido (se elimina por privacidad) |
| `CreditScore` | `score_crediticio` | Cuantitativa | Calificación de crédito (350 a 850; más alto = mejor historial) |
| `Geography` | `pais` | Cualitativa nominal | Francia, Alemania o España |
| `Gender` | `genero` | Cualitativa nominal | Hombre / Mujer |
| `Age` | `edad` | Cuantitativa discreta | Edad en años |
| `Tenure` | `antiguedad` | Cuantitativa discreta | Años como cliente del banco (0 a 10) |
| `Balance` | `saldo` | Cuantitativa continua | Saldo en la cuenta (€) |
| `NumOfProducts` | `num_productos` | Cualitativa ordinal | Productos contratados (1 a 4) |
| `HasCrCard` | `tarjeta_credito` | Binaria | 1 = tiene tarjeta de crédito |
| `IsActiveMember` | `miembro_activo` | Binaria | 1 = usa activamente sus cuentas |
| `EstimatedSalary` | `salario_estimado` | Cuantitativa continua | Salario anual estimado (€) |
| `Exited` | `abandono` | Binaria | **1 = abandonó el banco** |

> **Cuantitativa** = un número que se puede promediar (edad, saldo). **Cualitativa** = una categoría (país, género). **Binaria** = cualitativa con sólo dos opciones (sí/no).

## Paso 3 · Revisión de calidad

> 📄 **En el reporte PDF:** sección **2.1 Población y calidad de los datos** ("se revisó la calidad del dataset con 12 reglas... se cumplieron las 12").


Aunque el archivo parezca ordenado, se revisa con reglas explícitas. Cada regla tiene una **razón**:

| Revisión | Por qué importa |
|---|---|
| Valores faltantes | Un dato vacío no se puede usar en un cálculo |
| Filas duplicadas | Un cliente repetido tendría el doble de probabilidad de entrar a la muestra |
| Rangos válidos | Una edad de 150 años o un saldo negativo serían errores de captura |
| Categorías válidas | Escrituras distintas ("France", "france") se contarían como países diferentes |

In [4]:
# QUÉ HACE ESTA CELDA: Revisión de calidad: celdas vacías, filas repetidas e identificador único
# isna() marca con True cada celda vacía
celdas_vacias = datos.isna()
# sum() cuenta los True de cada columna
faltantes_por_columna = celdas_vacias.sum()
# suma todas las columnas: total de celdas vacías
total_faltantes = faltantes_por_columna.sum()
print("Valores faltantes:", total_faltantes)

# duplicated() marca True en cada fila que es copia de otra anterior
filas_repetidas = datos.duplicated()
print("Filas duplicadas:", filas_repetidas.sum())

# is_unique es True si ningún valor se repite
print("CustomerId único:", datos["CustomerId"].is_unique)

Valores faltantes: 0
Filas duplicadas: 0
CustomerId único: True


In [5]:
# QUÉ HACE ESTA CELDA: Revisión de calidad: que cada número esté dentro de un rango posible
# between(a, b) es True si el valor está entre a y b (incluidos)
edad_valida = datos["Age"].between(18, 100)
# all() es True sólo si TODAS las filas cumplen la regla
print("Edad entre 18 y 100:", edad_valida.all())

score_valido = datos["CreditScore"].between(300, 850)
print("CreditScore entre 300 y 850:", score_valido.all())

antiguedad_valida = datos["Tenure"].between(0, 10)
print("Antigüedad entre 0 y 10 años:", antiguedad_valida.all())

# True si el saldo es 0 o más
saldo_valido = datos["Balance"] >= 0
print("Saldo no negativo:", saldo_valido.all())

# True si el salario es mayor que 0
salario_valido = datos["EstimatedSalary"] > 0
print("Salario positivo:", salario_valido.all())

productos_validos = datos["NumOfProducts"].between(1, 4)
print("Productos entre 1 y 4:", productos_validos.all())

Edad entre 18 y 100: True
CreditScore entre 300 y 850: True
Antigüedad entre 0 y 10 años: True
Saldo no negativo: True
Salario positivo: True
Productos entre 1 y 4: True


In [6]:
# QUÉ HACE ESTA CELDA: Revisión de calidad: que las columnas de categorías sólo tengan valores permitidos
# isin([...]) es True si el valor es uno de los de la lista
tarjeta_valida = datos["HasCrCard"].isin([0, 1])
# all() es True sólo si TODAS las filas cumplen la regla
print("HasCrCard sólo 0 o 1:", tarjeta_valida.all())

activo_valido = datos["IsActiveMember"].isin([0, 1])
print("IsActiveMember sólo 0 o 1:", activo_valido.all())

abandono_valido = datos["Exited"].isin([0, 1])
print("Exited sólo 0 o 1:", abandono_valido.all())

pais_valido = datos["Geography"].isin(["France", "Germany", "Spain"])
print("País en {France, Germany, Spain}:", pais_valido.all())

genero_valido = datos["Gender"].isin(["Male", "Female"])
print("Género en {Male, Female}:", genero_valido.all())

HasCrCard sólo 0 o 1: True
IsActiveMember sólo 0 o 1: True
Exited sólo 0 o 1: True
País en {France, Germany, Spain}: True
Género en {Male, Female}: True


**Resultado:** el dataset **cumple todas las reglas**. No tiene faltantes, duplicados ni valores imposibles. Es lo esperable en un dataset público de referencia que ya fue depurado por quien lo publicó.

### Un caso que parece error pero no lo es: saldos en cero

> 📄 **En el reporte PDF:** sección 2.1: "El 36 % de los clientes tiene saldo cero; se decidió conservarlos".



In [7]:
# QUÉ HACE ESTA CELDA: Calcula el % de clientes con saldo exactamente 0 (reporte 2.1: el 36 %)
# == 0 da True si el saldo es exactamente 0
saldo_cero = datos["Balance"] == 0
# el promedio de una columna de 0 y 1 (o True y False) es la proporción; × 100 da el porcentaje
print("% de clientes con saldo exactamente 0:", saldo_cero.mean() * 100)

% de clientes con saldo exactamente 0: 36.17


📌 **Decisión:** los saldos en cero **se conservan**. No son datos faltantes: un cliente puede tener la cuenta abierta sin dinero (por ejemplo, si sólo usa la tarjeta de crédito). Borrarlos o "rellenarlos" cambiaría la realidad de la cartera. Esto sí se tendrá en cuenta al interpretar el saldo promedio, porque estos ceros hacen que la distribución tenga dos grupos (cuaderno 02).

## Paso 4 · Preparación

> 📄 **En el reporte PDF:** inicio de la sección **3** ("se eliminaron el número de renglón y el apellido, se tradujeron columnas... y se creó la variable grupo de edad") y criterios de inclusión de las secciones **2.1** y **2.4**.


📌 **Decisiones:**
1. **Eliminar `RowNumber`** (sólo es el número de renglón) y **`Surname`** (dato personal innecesario → minimización de datos del RGPD).
2. **Renombrar columnas al español** para que el análisis sea fácil de leer.
3. **Traducir categorías** (France → Francia, Female → Mujer).
4. **Crear `grupo_edad`**, rangos de edad útiles para tablas de frecuencia.

### ¿Quién entra al estudio? (criterios de inclusión)

Los **criterios de inclusión** son las reglas que dicen qué clientes cuentan para el estudio; los de **exclusión**, cuáles se dejan fuera.
| Criterio | ¿Cuántos lo cumplen? |
|---|---|
| Persona mayor de edad (≥ 18 años) | Todos (la edad mínima es 18) |
| Estatus de abandono conocido | Todos (sin faltantes) |
| Registro único (no duplicado) | Todos |

Por lo tanto, **los 10,000 clientes forman la población** del estudio, es decir, *todo el grupo* sobre el que queremos sacar conclusiones.

In [8]:
# QUÉ HACE ESTA CELDA: Borra 2 columnas y pone los nombres de las columnas en español
# RowNumber no aporta información y Surname es un dato personal innecesario (RGPD)
# drop borra esas columnas
poblacion = datos.drop(columns=["RowNumber", "Surname"])
# rename cambia el nombre de cada columna: {nombre viejo: nombre nuevo}
poblacion = poblacion.rename(columns={
    "CustomerId": "id_cliente",
    "CreditScore": "score_crediticio",
    "Geography": "pais",
    "Gender": "genero",
    "Age": "edad",
    "Tenure": "antiguedad",
    "Balance": "saldo",
    "NumOfProducts": "num_productos",
    "HasCrCard": "tarjeta_credito",
    "IsActiveMember": "miembro_activo",
    "EstimatedSalary": "salario_estimado",
    "Exited": "abandono",
})
print("Columnas:", list(poblacion.columns))

Columnas: ['id_cliente', 'score_crediticio', 'pais', 'genero', 'edad', 'antiguedad', 'saldo', 'num_productos', 'tarjeta_credito', 'miembro_activo', 'salario_estimado', 'abandono']


In [9]:
# QUÉ HACE ESTA CELDA: Traduce países y géneros al español y crea la columna grupo_edad
# map cambia cada valor según el diccionario {valor viejo: valor nuevo}
poblacion["pais"] = poblacion["pais"].map({"France": "Francia", "Germany": "Alemania", "Spain": "España"})
poblacion["genero"] = poblacion["genero"].map({"Male": "Hombre", "Female": "Mujer"})
# pd.cut corta la edad en rangos; [17, 29, 39, ...] quiere decir: más de 17 y hasta 29, más de 29 y hasta 39, ...
poblacion["grupo_edad"] = pd.cut(poblacion["edad"], [17, 29, 39, 49, 59, 100],
                                 labels=["18-29", "30-39", "40-49", "50-59", "60+"])

In [10]:
# QUÉ HACE ESTA CELDA: Comprobación de que la preparación salió bien
# comprobación del paso 4
print("Clientes en la población:", len(poblacion))
# cuenta los vacíos: si map no encuentra una traducción, deja la celda vacía
print("Países sin traducir:", poblacion["pais"].isna().sum())
print("Géneros sin traducir:", poblacion["genero"].isna().sum())
# in revisa si ese nombre está en la lista de columnas
print("¿Sigue la columna Surname?:", "Surname" in poblacion.columns)

Clientes en la población: 10000
Países sin traducir: 0
Géneros sin traducir: 0
¿Sigue la columna Surname?: False


In [11]:
# QUÉ HACE ESTA CELDA: Guarda la población preparada en un CSV para los cuadernos 02 y 03
# to_csv guarda la tabla en un archivo CSV; index=False no guarda el número de fila
poblacion.to_csv("../data/processed/poblacion_clientes.csv", index=False, encoding="utf-8-sig")
print("Población guardada, N =", len(poblacion))

Población guardada, N =

 10000


In [12]:
# QUÉ HACE ESTA CELDA: Muestra las primeras filas de la población ya preparada
# head() muestra sólo las primeras filas
poblacion.head()

,id_cliente,score_crediticio,pais,genero,edad,antiguedad,saldo,num_productos,tarjeta_credito,miembro_activo,salario_estimado,abandono,grupo_edad
0,15634602,619,Francia,Mujer,42,2,0.00,1,1,1,101348.88,1,40-49
1,15647311,608,España,Mujer,41,1,83807.86,1,0,1,112542.58,0,40-49
2,15619304,502,Francia,Mujer,42,8,159660.80,3,1,0,113931.57,1,40-49
3,15701354,699,Francia,Mujer,39,1,0.00,2,0,0,93826.63,0,30-39
4,15737888,850,España,Mujer,43,2,125510.82,1,1,1,79084.10,0,40-49


## Resumen

- Se trabaja con un **conjunto de datos (dataset) público de clientes de un banco europeo** (Francia, Alemania y España), uno de los más usados para estudiar la deserción bancaria.
- El dataset pasó **las 12 revisiones de calidad**; los saldos en cero se conservan porque son reales.
- Se eliminó el apellido por privacidad y las columnas quedaron en español.
- La **población de estudio** son los **N = 10,000 clientes**, guardados en `data/processed/poblacion_clientes.csv`.

➡️ **Siguiente cuaderno:** `02_muestreo_y_descriptiva.ipynb`: planteamiento del problema, selección de la muestra y estadística descriptiva.